In [34]:
# =============================================================================
# 03_feature_engineering.py
# Feature Engineering — TF-IDF, Stylometric, GloVe, RoBERTa
#
# =============================================================================
# Run order:  01_hc3_data_engineering.py -> 02_daigt_data_engineering.py -> this file
# Requires:   train.csv, val.csv, test.csv  (output of file 02)
# =============================================================================
#
# **Outputs produced:**
# - `tfidf_vectorizer.pkl`, `X_train_tfidf.npz`, `X_val_tfidf.npz`, `X_test_tfidf.npz`
# - `stylometric_features.csv` (train/val/test stylometric columns)
# - `glove_tokenizer.pkl`, `X_train_glove.npy`, `X_val_glove.npy`, `X_test_glove.npy`
# - `roberta_train.pt`, `roberta_val.pt`, `roberta_test.pt` (tokenised tensors)
#
# ---

import os
import re
import time
import string
import joblib
import numpy as np
import pandas as pd
from scipy import sparse
 
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
 
from sklearn.feature_extraction.text import TfidfVectorizer
 
for pkg in ['stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(pkg, quiet=True)
 
# ── Reproducibility — must match files 01 and 02 ──────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
 
# ── File paths — identical DATA_DIR convention used in file 02 ───────────────
DATA_DIR   = "/Users/yashaswini11/Desktop/Team_project/project"
TRAIN_FILE = os.path.join(DATA_DIR, "train.csv")
VAL_FILE   = os.path.join(DATA_DIR, "val.csv")
TEST_FILE  = os.path.join(DATA_DIR, "test.csv")
 
os.makedirs(DATA_DIR, exist_ok=True)
 
# ── TF-IDF settings ────────────────────────────────────────────────────────────
TFIDF_MAX_FEATURES = 50_000
TFIDF_NGRAM_RANGE   = (1, 2)
TFIDF_MIN_DF        = 3
 
# ── GloVe settings ────────────────────────────────────────────────────────────
GLOVE_DIM      = 100
GLOVE_MAX_LEN  = 200
# Download glove.6B.100d.txt from https://nlp.stanford.edu/projects/glove/
# and place it at GLOVE_PATH before running Section 6.
GLOVE_PATH     = os.path.join(DATA_DIR, "glove.6B.100d.txt")
 
# ── RoBERTa settings ───────────────────────────────────────────────────────────
ROBERTA_MODEL_NAME = "roberta-base"
ROBERTA_MAX_LENGTH = 256
 
print("Constants defined.")
print(f"  DATA_DIR : {DATA_DIR}")
print(f"  TF-IDF   : max_features={TFIDF_MAX_FEATURES}, ngrams={TFIDF_NGRAM_RANGE}")
print(f"  GloVe    : dim={GLOVE_DIM}, max_len={GLOVE_MAX_LEN}")
print(f"  RoBERTa  : {ROBERTA_MODEL_NAME}, max_length={ROBERTA_MAX_LENGTH}")

Constants defined.
  DATA_DIR : /Users/yashaswini11/Desktop/Team_project/project
  TF-IDF   : max_features=50000, ngrams=(1, 2)
  GloVe    : dim=100, max_len=200
  RoBERTa  : roberta-base, max_length=256


In [24]:
# ---
# ## Step 1 — Load Train / Val / Test Splits
#
# Loads the CSVs produced by `02_daigt_data_engineering.py`.  
# Columns: `text`, `label` (0=Human, 1=AI), `source`, `dataset`, `word_count`
 
print("=" * 60)
print("STEP 1: Loading train / val / test splits")
print("=" * 60)
 
for path in [TRAIN_FILE, VAL_FILE, TEST_FILE]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"\n[ERROR] {path} not found.\n"
            "Please run 01_hc3_data_engineering.py and 02_daigt_data_engineering.py first."
        )
 
train_df = pd.read_csv(TRAIN_FILE, encoding='utf-8-sig')
val_df   = pd.read_csv(VAL_FILE,   encoding='utf-8-sig')
test_df  = pd.read_csv(TEST_FILE,  encoding='utf-8-sig')
 
# Enforce types — CSV loading can read label as float if any NaN was present
for df in [train_df, val_df, test_df]:
    df['text']  = df['text'].astype(str)
    df['label'] = df['label'].astype(int)
 
print(f"  Train : {len(train_df):,} rows")
print(f"  Val   : {len(val_df):,} rows")
print(f"  Test  : {len(test_df):,} rows")
print(f"\n  Columns: {train_df.columns.tolist()}")
 
X_train_raw = train_df['text']
X_val_raw   = val_df['text']
X_test_raw  = test_df['text']
y_train     = train_df['label'].values
y_val       = val_df['label'].values
y_test      = test_df['label'].values
 
print(f"\n  Train label distribution:")
for label, count in train_df['label'].value_counts().sort_index().items():
    name = "Human" if label == 0 else "AI"
    print(f"    {label} ({name}) : {count:,}  ({count/len(train_df)*100:.1f}%)")
 
 

STEP 1: Loading train / val / test splits
  Train : 106,605 rows
  Val   : 22,844 rows
  Test  : 22,844 rows

  Columns: ['text', 'label', 'source', 'dataset', 'word_count']

  Train label distribution:
    0 (Human) : 55,923  (52.5%)
    1 (AI) : 50,682  (47.5%)


In [25]:
# ---
# ## Step 2 — Text Preprocessing (for TF-IDF only)
#
# Note: GloVe and RoBERTa pipelines use their own tokenisation later  
# (Steps 6 and 7) — this cleaning is specific to the TF-IDF / classical ML path.
#
# Steps:
# 1. Lowercase
# 2. Remove URLs, emails, digits
# 3. Remove punctuation (keep alphabetic only)
# 4. Tokenise, remove stopwords
# 5. Lemmatise
 
print("\n" + "=" * 60)
print("STEP 2: Text preprocessing for TF-IDF")
print("=" * 60)
 
STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
 
def preprocess(text: str) -> str:
    """
    Full NLP preprocessing pipeline for TF-IDF vectorisation.
 
    Args:
        text: Raw input string.
    Returns:
        Cleaned, lemmatised string with stopwords removed.
    """
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|\S+@\S+', '', text)   # remove URLs / emails
    text = re.sub(r'\d+', '', text)                          # remove digits
    text = re.sub(r'[^a-z\s]', ' ', text)                   # keep letters only
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)
 
# Smoke test
sample = 'The AI-generated text was surprisingly convincing, but experts identified it quickly.'
print('Original :', sample)
print('Processed:', preprocess(sample))
 
print('\nPreprocessing all splits... (may take 1-2 minutes)')
t0 = time.time()
X_train_clean = X_train_raw.apply(preprocess).reset_index(drop=True)
X_val_clean   = X_val_raw.apply(preprocess).reset_index(drop=True)
X_test_clean  = X_test_raw.apply(preprocess).reset_index(drop=True)
print(f'Done in {time.time() - t0:.1f}s')
 
# Index reset to default 0..n-1 above — ensures numpy boolean masks
# (e.g. dataset == 'HC3') align correctly with these Series in later notebooks.
print('Index reset on all clean Series — safe for numpy boolean masking.')


STEP 2: Text preprocessing for TF-IDF
Original : The AI-generated text was surprisingly convincing, but experts identified it quickly.
Processed: ai generated text surprisingly convincing expert identified quickly

Preprocessing all splits... (may take 1-2 minutes)
Done in 35.7s
Index reset on all clean Series — safe for numpy boolean masking.


In [26]:
# ---
# ## Step 3 — TF-IDF Feature Extraction
#
# Fitted on the **training set only** to prevent data leakage.  
# Unigrams + bigrams, max 50,000 features, sublinear TF scaling.
 
print("\n" + "=" * 60)
print("STEP 3: TF-IDF feature extraction")
print("=" * 60)
 
tfidf = TfidfVectorizer(
    ngram_range   = TFIDF_NGRAM_RANGE,
    max_features  = TFIDF_MAX_FEATURES,
    sublinear_tf  = True,
    min_df        = TFIDF_MIN_DF,
    strip_accents = 'unicode',
    analyzer      = 'word'
)
 
print('Fitting TF-IDF on training set only...')
t0 = time.time()
X_train_tfidf = tfidf.fit_transform(X_train_clean)   # fit + transform on train
X_val_tfidf   = tfidf.transform(X_val_clean)          # transform only — no leakage
X_test_tfidf  = tfidf.transform(X_test_clean)         # transform only — no leakage
print(f'Done in {time.time() - t0:.1f}s')
print(f'Matrix shape — Train: {X_train_tfidf.shape}  Val: {X_val_tfidf.shape}  Test: {X_test_tfidf.shape}')
print(f'Vocabulary size: {len(tfidf.vocabulary_):,}')
 


STEP 3: TF-IDF feature extraction
Fitting TF-IDF on training set only...
Done in 18.8s
Matrix shape — Train: (106605, 50000)  Val: (22844, 50000)  Test: (22844, 50000)
Vocabulary size: 50,000


In [27]:
# ---
# ## Step 4 — Stylometric Feature Extraction
#
# Hand-crafted document-level features used in the Hybrid CNN's Stream A  
# and for EDA / interpretability in the report:
# - `word_count` — total words
# - `char_count` — total characters
# - `avg_sentence_len` — mean words per sentence
# - `punct_density` — fraction of characters that are punctuation
# - `type_token_ratio` — vocabulary diversity (unique words / total words)
 
print("\n" + "=" * 60)
print("STEP 4: Stylometric feature extraction")
print("=" * 60)
 
def avg_sentence_length(text: str) -> float:
    """Mean word count per sentence."""
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    if not sentences:
        return 0.0
    return round(sum(len(s.split()) for s in sentences) / len(sentences), 2)
 
def punctuation_density(text: str) -> float:
    """Fraction of characters that are punctuation."""
    if not text:
        return 0.0
    punct_chars = sum(1 for c in text if c in string.punctuation)
    return round(punct_chars / len(text), 4)
 
def type_token_ratio(text: str) -> float:
    """Vocabulary diversity: unique words / total words."""
    tokens = text.lower().split()
    if not tokens:
        return 0.0
    return round(len(set(tokens)) / len(tokens), 4)
 
 
def build_stylometric_features(raw_text_series: pd.Series) -> pd.DataFrame:
    """
    Build a stylometric feature dataframe from raw (uncleaned) text.
 
    Uses RAW text, not the TF-IDF-cleaned text, because punctuation and
    capitalisation carry real stylometric signal that preprocessing removes.
 
    Args:
        raw_text_series: pandas Series of raw text strings.
    Returns:
        DataFrame with columns: word_count, char_count, avg_sentence_len,
        punct_density, type_token_ratio.
    """
    df = pd.DataFrame()
    df['word_count']       = raw_text_series.str.split().str.len()
    df['char_count']       = raw_text_series.str.len()
    df['avg_sentence_len'] = raw_text_series.apply(avg_sentence_length)
    df['punct_density']    = raw_text_series.apply(punctuation_density)
    df['type_token_ratio'] = raw_text_series.apply(type_token_ratio)
    return df
 
 
print('Building stylometric features from raw text...')
t0 = time.time()
style_train = build_stylometric_features(X_train_raw.reset_index(drop=True))
style_val   = build_stylometric_features(X_val_raw.reset_index(drop=True))
style_test  = build_stylometric_features(X_test_raw.reset_index(drop=True))
print(f'Done in {time.time() - t0:.1f}s')
 
print(f"\n  Stylometric feature columns: {style_train.columns.tolist()}")
print(f"\n  Average stats by label (train):")
style_train_labelled = style_train.copy()
style_train_labelled['label'] = y_train
stats = style_train_labelled.groupby('label')[
    ['word_count', 'avg_sentence_len', 'punct_density', 'type_token_ratio']
].mean().round(3)
stats.index = ['Human (0)', 'AI (1)']
print(stats.to_string())


STEP 4: Stylometric feature extraction
Building stylometric features from raw text...
Done in 9.7s

  Stylometric feature columns: ['word_count', 'char_count', 'avg_sentence_len', 'punct_density', 'type_token_ratio']

  Average stats by label (train):
           word_count  avg_sentence_len  punct_density  type_token_ratio
Human (0)     233.574            20.037          0.028             0.617
AI (1)        296.620            20.681          0.022             0.524


In [28]:
# ---
# ## Step 5 — Combined Feature Matrix (TF-IDF + Stylometric)
#
# Optional combined matrix for classical ML models that benefit from both  
# signal types. The standalone TF-IDF matrix is used for the baseline LR/RF/SVM  
# models in `04_classical_ml.ipynb`; this combined version is available for  
# ablation experiments.
 
print("\n" + "=" * 60)
print("STEP 5: Combining TF-IDF + stylometric features")
print("=" * 60)
 
from sklearn.preprocessing import StandardScaler
 
# Scale stylometric features — fit on train only, same leakage-prevention rule as TF-IDF
scaler = StandardScaler()
style_train_scaled = scaler.fit_transform(style_train)
style_val_scaled   = scaler.transform(style_val)
style_test_scaled  = scaler.transform(style_test)
 
# Horizontally stack sparse TF-IDF matrix with dense stylometric matrix
X_train_combined = sparse.hstack([X_train_tfidf, sparse.csr_matrix(style_train_scaled)]).tocsr()
X_val_combined    = sparse.hstack([X_val_tfidf,   sparse.csr_matrix(style_val_scaled)]).tocsr()
X_test_combined   = sparse.hstack([X_test_tfidf,  sparse.csr_matrix(style_test_scaled)]).tocsr()
 
print(f'  Combined matrix shape — Train: {X_train_combined.shape}')
print(f'  Combined matrix shape — Val  : {X_val_combined.shape}')
print(f'  Combined matrix shape — Test : {X_test_combined.shape}')
print(f'  (TF-IDF features: {X_train_tfidf.shape[1]:,} + stylometric: {style_train.shape[1]})')


STEP 5: Combining TF-IDF + stylometric features
  Combined matrix shape — Train: (106605, 50005)
  Combined matrix shape — Val  : (22844, 50005)
  Combined matrix shape — Test : (22844, 50005)
  (TF-IDF features: 50,000 + stylometric: 5)


In [29]:
# ---
# ## Step 6 — GloVe Tokenisation (for Hybrid CNN)
#
# Builds the integer-sequence input required by the Hybrid CNN's embedding stream.
#
# **Before running this cell:** download `glove.6B.100d.txt` from  
# https://nlp.stanford.edu/projects/glove/ and place it at the path in `GLOVE_PATH`.
#
# Steps:
# 1. Build a word-to-index vocabulary from the training set
# 2. Convert each text to a sequence of integer token IDs
# 3. Pad/truncate every sequence to `GLOVE_MAX_LEN`
# 4. Build an embedding matrix mapping each vocabulary word to its GloVe vector
 
print("\n" + "=" * 60)
print("STEP 6: GloVe tokenisation")
print("=" * 60)
 
if not os.path.exists(GLOVE_PATH):
    print(f"[WARNING] {GLOVE_PATH} not found.")
    print("Download glove.6B.100d.txt from https://nlp.stanford.edu/projects/glove/")
    print("and place it at the path above before running the Hybrid CNN notebook.")
    print("Skipping GloVe tokenisation for now — this cell can be re-run later.")
    GLOVE_AVAILABLE = False
else:
    GLOVE_AVAILABLE = True
    print(f"Found GloVe file at {GLOVE_PATH}")
 
def simple_tokenize(text: str) -> list:
    """Lowercase whitespace tokeniser — sufficient for GloVe lookup."""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.split()
 
 
def build_vocab(text_series: pd.Series, max_vocab_size: int = 30_000) -> dict:
    """
    Build a word-to-index vocabulary from training text only.
 
    Args:
        text_series:    Raw text Series (training set only — avoids leakage).
        max_vocab_size: Cap on vocabulary size, most frequent words kept.
    Returns:
        dict mapping word -> integer index. Index 0 reserved for padding,
        index 1 reserved for unknown/out-of-vocabulary words.
    """
    from collections import Counter
    counter = Counter()
    for text in text_series:
        counter.update(simple_tokenize(text))
 
    most_common = counter.most_common(max_vocab_size - 2)  # reserve 0=PAD, 1=UNK
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, _ in most_common:
        vocab[word] = len(vocab)
    return vocab
 
 
def texts_to_sequences(text_series: pd.Series, vocab: dict, max_len: int) -> np.ndarray:
    """
    Convert a Series of raw texts into a padded integer-sequence matrix.
 
    Args:
        text_series: Raw text Series.
        vocab:       Word-to-index dict from build_vocab().
        max_len:     Fixed sequence length — longer texts truncated,
                     shorter texts padded with 0 at the end.
    Returns:
        numpy array of shape (n_samples, max_len), dtype int32.
    """
    sequences = np.zeros((len(text_series), max_len), dtype=np.int32)
    for i, text in enumerate(text_series):
        tokens = simple_tokenize(text)[:max_len]
        for j, tok in enumerate(tokens):
            sequences[i, j] = vocab.get(tok, vocab['<UNK>'])  # UNK for OOV words
    return sequences
 
 
if GLOVE_AVAILABLE:
    print('Building vocabulary from training set...')
    glove_vocab = build_vocab(X_train_raw.reset_index(drop=True), max_vocab_size=30_000)
    print(f'  Vocabulary size: {len(glove_vocab):,}')
 
    print('\nConverting texts to padded integer sequences...')
    t0 = time.time()
    X_train_glove = texts_to_sequences(X_train_raw.reset_index(drop=True), glove_vocab, GLOVE_MAX_LEN)
    X_val_glove   = texts_to_sequences(X_val_raw.reset_index(drop=True),   glove_vocab, GLOVE_MAX_LEN)
    X_test_glove  = texts_to_sequences(X_test_raw.reset_index(drop=True),  glove_vocab, GLOVE_MAX_LEN)
    print(f'Done in {time.time() - t0:.1f}s')
    print(f'  Sequence shape — Train: {X_train_glove.shape}  Val: {X_val_glove.shape}  Test: {X_test_glove.shape}')
else:
    print('[SKIPPED] GloVe file not found — sequences not built.')
 
def load_glove_embeddings(glove_path: str, vocab: dict, embedding_dim: int) -> np.ndarray:
    """
    Build an embedding matrix aligned with the vocabulary index.
 
    Args:
        glove_path:    Path to glove.6B.<dim>d.txt.
        vocab:         Word-to-index dict from build_vocab().
        embedding_dim: Dimensionality of the GloVe vectors (100 for glove.6B.100d).
    Returns:
        numpy array of shape (len(vocab), embedding_dim). Words not found in
        GloVe (including <PAD> and <UNK>) are initialised to zero vectors.
    """
    embedding_matrix = np.zeros((len(vocab), embedding_dim), dtype=np.float32)
    found = 0
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip().split(' ')
            word  = parts[0]
            if word in vocab:
                vector = np.asarray(parts[1:], dtype=np.float32)
                embedding_matrix[vocab[word]] = vector
                found += 1
    return embedding_matrix, found
 
 
if GLOVE_AVAILABLE:
    print('Loading GloVe vectors and building embedding matrix...')
    t0 = time.time()
    glove_embedding_matrix, n_found = load_glove_embeddings(GLOVE_PATH, glove_vocab, GLOVE_DIM)
    print(f'Done in {time.time() - t0:.1f}s')
    print(f'  Embedding matrix shape : {glove_embedding_matrix.shape}')
    print(f'  Words found in GloVe   : {n_found:,} / {len(glove_vocab):,} '
          f'({n_found/len(glove_vocab)*100:.1f}%)')
else:
    print('[SKIPPED] GloVe embedding matrix not built.')


STEP 6: GloVe tokenisation
Found GloVe file at /Users/yashaswini11/Desktop/Team_project/project/glove.6B.100d.txt
Building vocabulary from training set...
  Vocabulary size: 30,000

Converting texts to padded integer sequences...
Done in 4.5s
  Sequence shape — Train: (106605, 200)  Val: (22844, 200)  Test: (22844, 200)
Loading GloVe vectors and building embedding matrix...
Done in 1.4s
  Embedding matrix shape : (30000, 100)
  Words found in GloVe   : 28,205 / 30,000 (94.0%)


In [30]:
! pip install torch --index-url https://download.pytorch.org/whl/cpu
! pip install transformers

Looking in indexes: https://download.pytorch.org/whl/cpu

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [31]:
# ---
# ## Step 7 — RoBERTa Tokenisation
#
# FIX: return_tensors changed from 'pt' to 'np'
# -----------------------------------------------
# Why it failed:
#   return_tensors='pt' makes HuggingFace call torch internally.
#   Torch 2.3.0 is installed in system Python but the VS Code notebook
#   kernel cannot resolve it at that internal call point → ImportError.
# Fix:
#   return_tensors='np' → tokenizer returns plain numpy arrays.
#   No torch needed locally at all.
#   On Colab (fine-tuning notebook), wrap with torch.tensor(np.load(...)).

print("\n" + "=" * 60)
print("STEP 7: RoBERTa tokenisation")
print("=" * 60)

try:
    from transformers import RobertaTokenizerFast
    ROBERTA_AVAILABLE = True
except ImportError:
    print('[WARNING] transformers not installed. Run: pip install transformers')
    ROBERTA_AVAILABLE = False

if ROBERTA_AVAILABLE:
    print(f'Loading tokenizer: {ROBERTA_MODEL_NAME}...')
    roberta_tokenizer = RobertaTokenizerFast.from_pretrained(ROBERTA_MODEL_NAME)

    def tokenize_for_roberta(text_series, labels):
        """
        Tokenise texts for RoBERTa. Returns numpy arrays (no torch needed).
        In Colab training notebook convert with: torch.tensor(np.load(...))
        """
        encodings = roberta_tokenizer(
            text_series.tolist(),
            truncation=True,
            padding='max_length',
            max_length=ROBERTA_MAX_LENGTH,
            return_tensors='np'          # ← was 'pt' — fixed to 'np'
        )
        return {
            'input_ids':      encodings['input_ids'],
            'attention_mask': encodings['attention_mask'],
            'labels':         labels                      # numpy int array
        }

    print('Tokenising train / val / test... (a few minutes for large datasets)')
    t0 = time.time()
    roberta_train = tokenize_for_roberta(X_train_raw.reset_index(drop=True), y_train)
    roberta_val   = tokenize_for_roberta(X_val_raw.reset_index(drop=True),   y_val)
    roberta_test  = tokenize_for_roberta(X_test_raw.reset_index(drop=True),  y_test)
    print(f'Done in {time.time() - t0:.1f}s')
    print(f'  input_ids shape — Train : {roberta_train["input_ids"].shape}')
    print(f'  input_ids shape — Val   : {roberta_val["input_ids"].shape}')
    print(f'  input_ids shape — Test  : {roberta_test["input_ids"].shape}')
else:
    print('[SKIPPED] RoBERTa tokenisation not run.')


STEP 7: RoBERTa tokenisation
Loading tokenizer: roberta-base...
Tokenising train / val / test... (a few minutes for large datasets)
Done in 28.0s
  input_ids shape — Train : (106605, 256)
  input_ids shape — Val   : (22844, 256)
  input_ids shape — Test  : (22844, 256)


In [32]:
# ---
# ## Step 8 — Save All Feature Artefacts

print("\n" + "=" * 60)
print("STEP 8: Saving feature artefacts")
print("=" * 60)

# TF-IDF
joblib.dump(tfidf, os.path.join(DATA_DIR, 'tfidf_vectorizer.pkl'))
sparse.save_npz(os.path.join(DATA_DIR, 'X_train_tfidf.npz'), X_train_tfidf)
sparse.save_npz(os.path.join(DATA_DIR, 'X_val_tfidf.npz'),   X_val_tfidf)
sparse.save_npz(os.path.join(DATA_DIR, 'X_test_tfidf.npz'),  X_test_tfidf)
print('  Saved: tfidf_vectorizer.pkl, X_train/val/test_tfidf.npz')

# Combined TF-IDF + stylometric
sparse.save_npz(os.path.join(DATA_DIR, 'X_train_combined.npz'), X_train_combined)
sparse.save_npz(os.path.join(DATA_DIR, 'X_val_combined.npz'),   X_val_combined)
sparse.save_npz(os.path.join(DATA_DIR, 'X_test_combined.npz'),  X_test_combined)
joblib.dump(scaler, os.path.join(DATA_DIR, 'stylometric_scaler.pkl'))
print('  Saved: X_train/val/test_combined.npz, stylometric_scaler.pkl')

# Stylometric CSVs (raw, unscaled — for EDA and report)
style_train.assign(label=y_train).to_csv(os.path.join(DATA_DIR, 'stylometric_train.csv'), index=False)
style_val.assign(label=y_val).to_csv(os.path.join(DATA_DIR,   'stylometric_val.csv'),   index=False)
style_test.assign(label=y_test).to_csv(os.path.join(DATA_DIR, 'stylometric_test.csv'),  index=False)
print('  Saved: stylometric_train/val/test.csv')

# Cleaned text (avoids re-running preprocessing in later notebooks)
pd.DataFrame({'text': X_train_clean}).to_csv(os.path.join(DATA_DIR, 'X_train_clean.csv'), index=False)
pd.DataFrame({'text': X_val_clean}).to_csv(os.path.join(DATA_DIR,   'X_val_clean.csv'),   index=False)
pd.DataFrame({'text': X_test_clean}).to_csv(os.path.join(DATA_DIR,  'X_test_clean.csv'),  index=False)
print('  Saved: X_train/val/test_clean.csv')

# GloVe
if GLOVE_AVAILABLE:
    joblib.dump(glove_vocab, os.path.join(DATA_DIR, 'glove_vocab.pkl'))
    np.save(os.path.join(DATA_DIR, 'X_train_glove.npy'),          X_train_glove)
    np.save(os.path.join(DATA_DIR, 'X_val_glove.npy'),            X_val_glove)
    np.save(os.path.join(DATA_DIR, 'X_test_glove.npy'),           X_test_glove)
    np.save(os.path.join(DATA_DIR, 'glove_embedding_matrix.npy'), glove_embedding_matrix)
    print('  Saved: glove_vocab.pkl, X_train/val/test_glove.npy, glove_embedding_matrix.npy')
else:
    print('  [SKIPPED] GloVe artefacts not saved.')

# RoBERTa — saved as .npy (no torch needed here)
# Load on Colab with: torch.tensor(np.load('roberta_train_ids.npy'))
if ROBERTA_AVAILABLE:
    for split_name, split in [('train', roberta_train), ('val', roberta_val), ('test', roberta_test)]:
        np.save(os.path.join(DATA_DIR, f'roberta_{split_name}_ids.npy'),    split['input_ids'])
        np.save(os.path.join(DATA_DIR, f'roberta_{split_name}_mask.npy'),   split['attention_mask'])
        np.save(os.path.join(DATA_DIR, f'roberta_{split_name}_labels.npy'), split['labels'])
    print('  Saved: roberta_train/val/test _ids / _mask / _labels .npy')
    print('  On Colab: torch.tensor(np.load("roberta_train_ids.npy"))')
else:
    print('  [SKIPPED] RoBERTa artefacts not saved.')


STEP 8: Saving feature artefacts
  Saved: tfidf_vectorizer.pkl, X_train/val/test_tfidf.npz
  Saved: X_train/val/test_combined.npz, stylometric_scaler.pkl
  Saved: stylometric_train/val/test.csv
  Saved: X_train/val/test_clean.csv
  Saved: glove_vocab.pkl, X_train/val/test_glove.npy, glove_embedding_matrix.npy
  Saved: roberta_train/val/test _ids / _mask / _labels .npy
  On Colab: torch.tensor(np.load("roberta_train_ids.npy"))


In [33]:
# ---
# ## Step 9 — Final Summary

print("\n" + "=" * 60)
print("FEATURE ENGINEERING — COMPLETE")
print("=" * 60)
print()
print("  Feature sets:")
print(f"    TF-IDF only          : {X_train_tfidf.shape[1]:,} features")
print(f"    TF-IDF + stylometric : {X_train_combined.shape[1]:,} features")
print(f"    GloVe sequences      : {'built — shape ' + str(X_train_glove.shape) if GLOVE_AVAILABLE else 'SKIPPED'}")
print(f"    RoBERTa tokens       : {'built — shape ' + str(roberta_train['input_ids'].shape) if ROBERTA_AVAILABLE else 'SKIPPED'}")
print()
print("  Sample sizes:")
print(f"    Train : {len(train_df):,}")
print(f"    Val   : {len(val_df):,}")
print(f"    Test  : {len(test_df):,}")
print()
print("  Artefacts saved to:", DATA_DIR)
print()
print("  Next notebooks:")
print("    -> 04_classical_ml.ipynb       : LR / RF / SVM  (loads tfidf artefacts)")
print("    -> 05_hybrid_cnn.ipynb         : Hybrid CNN      (loads glove artefacts)")
print("    -> 06_roberta_finetuning.ipynb : RoBERTa         (loads roberta .npy on Colab)")
print()
print("=" * 60)


FEATURE ENGINEERING — COMPLETE

  Feature sets:
    TF-IDF only          : 50,000 features
    TF-IDF + stylometric : 50,005 features
    GloVe sequences      : built — shape (106605, 200)
    RoBERTa tokens       : built — shape (106605, 256)

  Sample sizes:
    Train : 106,605
    Val   : 22,844
    Test  : 22,844

  Artefacts saved to: /Users/yashaswini11/Desktop/Team_project/project

  Next notebooks:
    -> 04_classical_ml.ipynb       : LR / RF / SVM  (loads tfidf artefacts)
    -> 05_hybrid_cnn.ipynb         : Hybrid CNN      (loads glove artefacts)
    -> 06_roberta_finetuning.ipynb : RoBERTa         (loads roberta .npy on Colab)

